# Velocity UMAP Analysis for Embryonic Data

This notebook runs `plot_velocity_umap` analysis using the trained model from the Lamda5 experiment.

In [10]:
import os
import sys
import numpy as np
import torch
import scanpy as sc
import scvelo as scv
import scipy.sparse as sp
import matplotlib.pyplot as plt
from datetime import datetime

# Add scDiffusion directory to path
sys.path.insert(0, '/home/suzuki/Projects/scDiffusion')

from guided_diffusion.cell_model import Cell_Unet
from ODE.ode_analysis1106 import GeneODE, ODE_ML_Hybrid

print(f"PyTorch: {torch.__version__}")
print(f"Scanpy: {sc.__version__}")
print(f"Scvelo: {scv.__version__}")

PyTorch: 2.5.1
Scanpy: 1.9.3
Scvelo: 0.3.3


## Set up paths

In [3]:
# Paths
work_dir = '/home/suzuki/Projects/scDiffusion/work/20260215_embryonic/20260224_084326_Lamda5'
train_dir = os.path.join(work_dir, '20260224_084328_train')
model_dir = os.path.join(train_dir, 'checkpoints/pbmc68k_soft_20251127_hvg1024')

# Data and model paths
data_dir = '/home/suzuki/Projects/scDiffusion/work/20260215_embryonic/data/Embryonic.h5ad'
edge_tsv_path = '/home/suzuki/Projects/scDiffusion/external_data/Mouse_tf_target_edges.tsv'
model_path = os.path.join(model_dir, 'ema_0.9999_100000.pt')  # Best model
output_dir = os.path.join(work_dir, f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_velocity_analysis")

os.makedirs(output_dir, exist_ok=True)

print(f"Work directory: {work_dir}")
print(f"Model path: {model_path}")
print(f"Data path: {data_dir}")
print(f"Output directory: {output_dir}")
print(f"Files exist: model={os.path.exists(model_path)}, data={os.path.exists(data_dir)}")

Work directory: /home/suzuki/Projects/scDiffusion/work/20260215_embryonic/20260224_084326_Lamda5
Model path: /home/suzuki/Projects/scDiffusion/work/20260215_embryonic/20260224_084326_Lamda5/20260224_084328_train/checkpoints/pbmc68k_soft_20251127_hvg1024/ema_0.9999_100000.pt
Data path: /home/suzuki/Projects/scDiffusion/work/20260215_embryonic/data/Embryonic.h5ad
Output directory: /home/suzuki/Projects/scDiffusion/work/20260215_embryonic/20260224_084326_Lamda5/20260224_102141_velocity_analysis
Files exist: model=True, data=True


## Load preprocessed data

In [4]:
print(f"Loading data from {data_dir}...")
adata_real = sc.read_h5ad(data_dir)
print(f"Data shape: {adata_real.shape}")
print(f"Data obs columns: {list(adata_real.obs.columns)}")

# Set final annotation
if 'celltype' in adata_real.obs.columns:
    adata_real.obs['final_annotation'] = adata_real.obs['celltype'].astype(str)
    print(f"Cell types: {adata_real.obs['final_annotation'].unique()}")
else:
    print("Warning: 'celltype' column not found")

Loading data from /home/suzuki/Projects/scDiffusion/work/20260215_embryonic/data/Embryonic.h5ad...
Data shape: (156726, 1024)
Data obs columns: ['Age', 'celltype', 'Souporcell', 'Subclass', 'Superclass', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'assay_ontology_term_id', 'development_stage_ontology_term_id', 'donor_id', 'suspension_type', 'tissue_type', 'dissection', 'fraction_mitochondrial', 'fraction_unspliced', 'total_genes', 'total_UMIs', 'sample_id', 'CellType', 'cluster_id', 'cell_type_ontology_term_id', 'tissue_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'n_genes']
Cell types: ['Oligodendrocyte lineage' 'Interneuron' 'Neuron' 'Radial glia'
 'Neuroblast' 'Intermediate progenitor' 'Glioblast' 'Optic'
 'Rathkes pouch' 'Peripheral neuron' 'Peripheral neuroblast'
 'Neural crest progenitor' 'Schwann cell lineage' 'Endot

## Load trained model and compute velocities

In [5]:
print("Loading model and computing velocities...")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

gene_list = list(adata_real.var_names)
print(f"Number of genes: {len(gene_list)}")

# Load checkpoint
checkpoint = torch.load(model_path, map_location=device, weights_only=False)
state_dict = checkpoint.get('model_state_dict', checkpoint.get('model', checkpoint))
print(f"Checkpoint keys: {list(checkpoint.keys())[:5]}")

# Build model
ode = GeneODE(gene_list=gene_list, edge_tsv_path=edge_tsv_path, soft=True, device=device)
ml_model = Cell_Unet(input_dim=len(gene_list))
hybrid_model = ODE_ML_Hybrid(ode_model=ode, ml_model=ml_model, timesteps=1000).to(device)
hybrid_model.load_state_dict(state_dict, strict=False)
hybrid_model.eval()
print("Model loaded successfully")

Loading model and computing velocities...
Device: cuda
Number of genes: 1024
Checkpoint keys: ['ode_model.W', 'ode_model.b', 'ode_model.gamma', 'ode_model.mask', 'ode_model.scale']
ODEで使う遺伝子の数0, ODEで使う遺伝子の数1024
Model loaded successfully


## Compute velocity

In [6]:
# Extract expression matrix
X = adata_real.X
if sp.issparse(X):
    X = X.toarray()
X = np.asarray(X, dtype=np.float32)
print(f"Expression matrix shape: {X.shape}")

# Compute velocity
n = X.shape[0]
batch_size = 1024
V = np.zeros_like(X, dtype=np.float32)

print("Computing velocities in batches...")
with torch.no_grad():
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        xb = torch.from_numpy(X[start:end]).to(device)
        # Get velocity (dx/dt) from ODE model
        vb = hybrid_model.ode_model(xb).detach().cpu().numpy()
        V[start:end] = np.asarray(vb, dtype=np.float32)
        if (start // batch_size) % 5 == 0:
            print(f"  Processed {end}/{n} cells")

# Handle NaN values
V[~np.isfinite(V)] = 0.0
print(f"Velocity matrix shape: {V.shape}")
print(f"Velocity statistics - mean: {np.mean(V):.4f}, std: {np.std(V):.4f}")

Expression matrix shape: (156726, 1024)
Computing velocities in batches...
  Processed 1024/156726 cells
  Processed 6144/156726 cells
  Processed 11264/156726 cells
  Processed 16384/156726 cells
  Processed 21504/156726 cells
  Processed 26624/156726 cells
  Processed 31744/156726 cells
  Processed 36864/156726 cells
  Processed 41984/156726 cells
  Processed 47104/156726 cells
  Processed 52224/156726 cells
  Processed 57344/156726 cells
  Processed 62464/156726 cells
  Processed 67584/156726 cells
  Processed 72704/156726 cells
  Processed 77824/156726 cells
  Processed 82944/156726 cells
  Processed 88064/156726 cells
  Processed 93184/156726 cells
  Processed 98304/156726 cells
  Processed 103424/156726 cells
  Processed 108544/156726 cells
  Processed 113664/156726 cells
  Processed 118784/156726 cells
  Processed 123904/156726 cells
  Processed 129024/156726 cells
  Processed 134144/156726 cells
  Processed 139264/156726 cells
  Processed 144384/156726 cells
  Processed 149504/

## Set velocity in AnnData object

In [7]:
# Store velocity in layers
if "X" not in adata_real.layers:
    adata_real.layers["X"] = adata_real.X.copy()
adata_real.layers["velocity_ode"] = V

print(f"Layers in object: {list(adata_real.layers.keys())}")

Layers in object: ['X', 'velocity_ode']


## UMAP and Velocity graph computation

In [8]:
# Ensure UMAP coordinates exist
if "X_pca" not in adata_real.obsm:
    print("Computing PCA...")
    sc.tl.pca(adata_real, svd_solver='arpack', n_comps=50)

if "neighbors" not in adata_real.uns:
    print("Computing neighbors...")
    scv.pp.neighbors(adata_real, n_neighbors=15, n_pcs=40)

if "X_umap" not in adata_real.obsm:
    print("Computing UMAP...")
    sc.tl.umap(adata_real)

print(f"Embeddings in object: {list(adata_real.obsm.keys())}")

Computing PCA...
Computing neighbors...
computing neighbors
    finished (0:00:46) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
Computing UMAP...
Embeddings in object: ['X_UMAP', 'X_tSNE', 'X_pca', 'X_umap']


## Vis

In [ ]:
sc.pl.umap(adata_real, color=["celltype"])

In [ ]:
adata_real.obs

## Compute velocity graph (this may take a while)

In [13]:
print("Computing velocity graph...")
scv.tl.velocity_graph(
    adata_real,
    vkey="velocity_ode",
    xkey="X",
    backend="threading",
    n_jobs=64
)
print("Velocity graph computed successfully")

Computing velocity graph...
computing velocity graph (using 64/64 cores)


  0%|          | 0/156726 [00:00<?, ?cells/s]

KeyboardInterrupt: 

## Compute velocity embedding

In [ ]:
print("Computing velocity embedding...")
scv.tl.velocity_embedding(adata_real, basis="umap", vkey="velocity_ode")
print("Velocity embedding computed successfully")

## Plot velocity UMAP

In [ ]:
print("Generating velocity UMAP visualization...")

scv.pl.velocity_embedding_stream(
    adata_real,
    basis="umap",
    vkey="velocity_ode",
    color="final_annotation",
    legend_loc="right margin",
    show=False
)

save_path = os.path.join(output_dir, "velocity_umap.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved Velocity UMAP to {save_path}")

## Summary

In [ ]:
print("="*50)
print("Analysis Complete!")
print("="*50)
print(f"Output directory: {output_dir}")
print(f"Files saved:")
for f in os.listdir(output_dir):
    print(f"  - {f}")